In [ ]:
%matplotlib qt

import numpy as np
import hyperspy.api as hs
from hyperspy.signals import Signal2D
import gc

import pyxem.signals as pxm

In [ ]:
#Load the data
Data = hs.load(r"C:User\YourPathHere", reader="hspy")
Data.plot()

In [ ]:
#Sometimes significant bleedthrough can ruin individual patterns on the navigational axes
#This code attemps to find and replace these bleedthrough events with the mean of the pattern
#It should be used mindfully, as STEM images containing large contrasts in signal, may have those high intensity areas replaced.

TotalIntensity = Data.data.sum(axis=(-2, -1))

Median = np.median(TotalIntensity)
MedianAbsDeviation = np.median(np.abs(TotalIntensity - Median))

Threshold = Median + 13 * MedianAbsDeviation
Mask = TotalIntensity > Threshold

Mean = Data.data[~Mask].mean(axis=0)

# Replace bad pixels
Data.data[Mask] = Mean

gc.collect()
Data.plot()

In [ ]:
#Rebin if necessary
#Data = Data.rebin(scale=(1, 1, 2, 2))

In [ ]:
Data.set_signal_type('electron_diffraction')  # assign electron diffraction so pyxem can work
gc.collect()
Data.data = Data.data.astype('float32')
Data.data *= 1 / Data.data.max()

In [ ]:
import pyxem
print(pyxem.__version__)
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 14})

In [ ]:
import numpy as np
from hyperspy.interactive import interactive

#Defines a new function for the 2D diffraction class, which allows vmax to be passed
def plot_integrated_intensity_new1(
    self,
    roi,
    out_signal_axes=None,
    signal_vmax=None,
    signal_vmin=None,
    auto_signal_contrast=False,
    **kwargs,
):
    """Interactively plot integrated intensity inside an ROI."""

    # Plot diffraction signal if needed
    if self._plot is None or not self._plot.is_active:

        plot_kwargs = {}

        if auto_signal_contrast:
            plot_kwargs["vmin"] = np.nanpercentile(self.data, 1)
            plot_kwargs["vmax"] = np.nanpercentile(self.data, 99.5)

        if signal_vmin is not None:
            plot_kwargs["vmin"] = signal_vmin

        if signal_vmax is not None:
            plot_kwargs["vmax"] = signal_vmax

        self.plot(**plot_kwargs)

    # Slice diffraction signal using ROI:
    sliced_signal = roi.interactive(self, axes=self.axes_manager.signal_axes)

    # The output signal:
    out = self._get_sum_signal(self, out_signal_axes)
    out.metadata.General.title = "Integrated intensity"

    interactive(
        sliced_signal.nansum,
        axis=sliced_signal.axes_manager.signal_axes,
        event=roi.events.changed,
        recompute_out_event=None,
        out=out,
    )

    #The final plot
    out.plot(**kwargs)

from pyxem.signals import Diffraction2D

Diffraction2D.plot_integrated_intensity_new1 = plot_integrated_intensity_new1

In [ ]:
plt.close() #Clears any previously open plot to make the plotting function work properly

RegionOfInterest = hs.roi.CircleROI(cx=0.137, cy=0.025, r=0.005)
Data.plot_integrated_intensity_new1(RegionOfInterest, signal_vmax=0.01)